# 071 · Introduction to Transformers

Two things to verify here, both from the lesson:

1. The 2017 base model comes to **63,119,496** parameters — but only with
   **weight tying**. Untied it is 82,063,496.
2. The parameter share **inverts** relative to every earlier architecture in
   the course: 69.9% stack, 30% vocabulary.

Requires nothing but the standard library.

In [ ]:
# Architecture from "Attention Is All You Need", base configuration.
D, LAYERS, FF, VOCAB, HEADS = 512, 6, 2048, 37_000, 8

mha  = 4 * D * D + 4 * D            # Q, K, V and output projections
ffn  = D * FF + FF + FF * D + D     # two linear layers
norm = 2 * D                        # layer norm: gain + bias

encoder_layer = mha + ffn + 2 * norm
decoder_layer = 2 * mha + ffn + 3 * norm   # self-attention AND cross-attention

print(f"attention block      {mha:>12,}")
print(f"feed-forward block   {ffn:>12,}")
print(f"per encoder layer    {encoder_layer:>12,}")
print(f"per decoder layer    {decoder_layer:>12,}")

Note the decoder layer is bigger: it has *two* attention blocks, because it
attends both to itself and to the encoder.

In [ ]:
embedding = VOCAB * D
stack = LAYERS * (encoder_layer + decoder_layer)

untied = embedding + stack + D * VOCAB + VOCAB   # separate output projection
tied   = embedding + stack + VOCAB               # output shares the embedding

print(f"6 encoder layers     {LAYERS * encoder_layer:>12,}")
print(f"6 decoder layers     {LAYERS * decoder_layer:>12,}")
print(f"embeddings           {embedding:>12,}")
print()
print(f"untied total         {untied:>12,}")
print(f"tied total           {tied:>12,}   <- published is about 65,000,000")
print(f"tying saves          {untied - tied:>12,}")

## The share inverts

Every architecture measured earlier in the course had the vocabulary-facing
layers dominating. Check that this one does not.

In [ ]:
print(f"attention + FFN stack   {stack / tied:>6.1%}")
print(f"vocabulary-facing       {embedding / tied:>6.1%}")
print()
for name, share in [("MNIST ANN", .99), ("LeNet-5", .973), ("dog-vs-cat", .994),
                    ("VGG16", .894), ("next-word LSTM", .96), ("Sutskever 2014", .833)]:
    print(f"  {name:<16} interface {share:>6.1%}")
print(f"  {'Transformer':<16} interface {embedding / tied:>6.1%}   <- inverted")

## Path length

The claim is that any two positions are one step apart, against $O(n)$ for a
recurrent model. This is the whole reason the architecture exists.

In [ ]:
print(f"{'distance':>10}{'RNN steps':>12}{'transformer':>14}")
for n in (10, 100, 1_000, 10_000):
    print(f"{n:>10,}{n:>12,}{1:>14}")

print()
print("Two separate consequences:")
print("  1. all positions compute in parallel")
print("  2. one step means ONE multiplication, so no vanishing across the sequence")
print()
print("And the bill:")
for n in (10, 100, 1_000):
    print(f"  {n:>5} tokens -> {n*n:>9,} attention scores   (O(n^2))")

## Exercise

1. Recompute for the *big* configuration in the paper: `D=1024`, `LAYERS=6`,
   `FF=4096`, 16 heads. Published count is about 213M — how close do you get?
2. At what vocabulary size does the interface become dominant again? Solve for
   the `VOCAB` where `embedding == stack`.
3. Why does the number of heads not appear anywhere in this count?